# 08 · Evaluación en test (base y entrenado)

**Qué contiene.** Evalúa el agente sobre el conjunto de test (585 preguntas técnicas, 23 familias de datasets no vistas), una vez por modelo. `CORRIDA` elige `"base"` (prompt optimizado) o `"entrenado"` (sin prompt de sistema). Guarda cada 25 preguntas y retoma si se corta.

| | |
|---|---|
| **Runtime** | GPU T4 (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU) |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `catalogo_busqueda.csv`, `eval_test_final2.csv`, `specs_test_v2.csv`, `tools_v1.json`, `qlora_completo/adapter` (solo `entrenado`) |
| **Salidas** | `baseline_test_tecnica_completa_optimizado` y `finetuned_test_tecnica_completa_basico`: `.csv`, `_episodios.jsonl` y `_resultados.jsonl` |
| **Secretos de Colab** | `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` |
| **Costo y tiempo** | Sin costo de API. Unos 85 a 100 minutos por corrida (8,5 y 10,2 s por pregunta) |
| **Orden** | Después de `07`; antes de `09`. Correr la celda 0 y la celda 1 con cada valor de `CORRIDA` |
| **En el repositorio** | `scripts/evaluar.py` |

**Advertencias.**
- Cada modelo se evaluó una sola vez en test. No repetir ni hacer pruebas de humo sobre test (el notebook lo impide con `assert`).
- `CORRIDA` viene en `"base"`. Para el modelo entrenado, cambiarlo antes de correr y reiniciar el entorno.


In [ ]:
!pip -q install unsloth langfuse
print("CELDA 1 de 2 (TEST): PREPARACIÓN")
CORRIDA = "base"
import os
os.environ["UNSLOTH_DISABLE_CPU_FALLBACK"] = "1"
from unsloth import FastLanguageModel
import gc, json, shutil
import torch
import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
from google.colab import drive, userdata

assert CORRIDA in ("entrenado", "base"), "CORRIDA debe ser 'entrenado' o 'base'"
assert torch.cuda.is_available(), "No hay GPU: Runtime > Change runtime type > T4 GPU"
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
BASE_ID = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
ADAPTADOR = f"{RUTA}/qlora_completo/adapter"
MODELO_CARGADO = ADAPTADOR if CORRIDA == "entrenado" else BASE_ID
if CORRIDA == "entrenado":
    assert os.path.exists(f"{ADAPTADOR}/adapter_config.json"), f"No encuentro el adaptador en {ADAPTADOR}"
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO_CARGADO, max_seq_length=MAX_SEQ, load_in_4bit=True)
FastLanguageModel.for_inference(model)
assert model.max_seq_length == MAX_SEQ, f"El modelo quedó con max_seq_length={model.max_seq_length}"
print(f"Corrida: {CORRIDA} | modelo: {MODELO_CARGADO} | max_seq_length real: {model.max_seq_length} | "
      f"memoria de GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ---------- Langfuse (cuenta de EE.UU.) ----------
HOST = "https://us.cloud.langfuse.com"
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY").strip()
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY").strip()
os.environ["LANGFUSE_HOST"] = HOST
os.environ["LANGFUSE_BASE_URL"] = HOST
from langfuse import get_client
lf = get_client()
print("Langfuse auth_check:", lf.auth_check())

# ---------- Archivos de trabajo (copias locales) ----------
for nombre in ["cache_api.db", "busqueda.db"]:
    if not os.path.exists(f"/content/{nombre}"):
        shutil.copy(f"{RUTA}/{nombre}", f"/content/{nombre}")
tools = json.load(open(f"{RUTA}/tools_v1.json"))
print("Herramientas:", [t["function"]["name"] for t in tools])

In [ ]:
print("CELDA 2 de 2 (TEST): AGENTE + EVALUADOR + CORRIDA CON LANGFUSE Y RETOMA")
assert all(x in globals() for x in ["model", "tokenizer", "lf", "RUTA", "tools", "CORRIDA", "MODELO_CARGADO"]), \
    "Ejecutá antes la CELDA 1 de 2"
assert model.max_seq_length >= 4096, f"El modelo quedó con max_seq_length={model.max_seq_length}: hay que recargarlo"
assert ("qlora_completo" in MODELO_CARGADO) == (CORRIDA == "entrenado"), "El modelo cargado no coincide con CORRIDA"
import re, json, time, sqlite3, shutil, unicodedata, os
import logging, warnings
import numpy as np
import pandas as pd
import requests
import torch
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ---------- Configuración ----------
SPLIT = "test"          # "test" para la corrida final; "val" solo para la prueba de humo
ESTILO = "tecnica"
N_PRUEBA = None         # None = todo; 8 = prueba de humo (solo con SPLIT = "val")
MAX_PASOS = 6
MAX_NUEVOS = 300
assert not (SPLIT == "test" and N_PRUEBA), "No hagas pruebas de humo sobre test"
assert SPLIT == "test" or N_PRUEBA, "La validación completa ya está medida: no la repitas"
if CORRIDA == "entrenado":
    PROMPT, PREFIJO, ETIQUETA_MODELO = "basico", "finetuned", "qwen3-4b-instruct-2507-qlora"
else:
    PROMPT, PREFIJO, ETIQUETA_MODELO = "optimizado", "baseline", "qwen3-4b-instruct-2507-base"
CADA = 4 if N_PRUEBA else 25
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}
NOMBRE = f"{PREFIJO}_{SPLIT}_{ESTILO}_{'prueba' if N_PRUEBA else 'completa'}_{PROMPT}"
ARCH_DRIVE = f"{RUTA}/{NOMBRE}_resultados.jsonl"
ARCH_LOCAL = f"/content/{NOMBRE}_resultados.jsonl"
print(f"Corrida: {CORRIDA} | prompt: {PROMPT} | split: {SPLIT} | archivos: {NOMBRE}*")

PROMPT_OPT = """Sos un asistente que responde preguntas sobre estadísticas oficiales de Argentina usando tres herramientas. Seguí siempre este procedimiento:
1. Llamá a search_series con 2 a 6 palabras clave del tema (sin fechas ni números). Elegí, entre los candidatos devueltos, la serie cuyo título, distribución y frecuencia (mensual, trimestral o anual) mejor coincidan con lo que se pide. Usá solo ids devueltos por search_series; nunca inventes ids.
2. Llamá a get_series con la serie elegida:
- Valor de un período: start_date y end_date iguales, en formato AAAA (series anuales) o AAAA-MM.
- Último dato disponible: limit=1 y sort="desc", sin fechas.
- Variación interanual: representation_mode="percent_change_a_year_ago", start_date un año antes del período pedido y end_date el período pedido, con fechas completas AAAA-MM-DD (día 01; en series trimestrales, el primer mes del trimestre). La API devuelve una fracción (0.05 equivale a 5%) y la última fila corresponde al período pedido.
3. Llamá a submit_answer con la respuesta, el valor y la fecha del dato (formato AAAA-MM, o AAAA si la serie es anual). Para variaciones, el valor va en porcentaje (la fracción multiplicada por 100).
Si get_series devuelve data vacía, la API no tiene datos de esa serie para ese período: entregá submit_answer con valor y fecha en null y explicalo en la respuesta. No inventes valores."""
SISTEMA = PROMPT_OPT if PROMPT == "optimizado" else None

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
PARAMS_TOOL = {"ids", "start_date", "end_date", "representation_mode", "limit", "sort"}
cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def clave(params):
    p = {k: str(v) for k, v in params.items() if v is not None}
    return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

# ---------- Herramientas: caché primero, API real si falta ----------
class Herramientas:
    def __init__(self):
        self.cache = sqlite3.connect("/content/cache_api.db")
        self.fts = sqlite3.connect("/content/busqueda.db")
        self.fallos_cache, self.sin_red = 0, 0

    def search_series(self, query, k=10):
        toks = [t for t in re.findall(r"[a-z0-9]+", norm(query)) if t not in STOP and len(t) > 1]
        toks = list(dict.fromkeys(toks))
        if not toks:
            return "[]"
        partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
        rows = self.fts.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                                "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
        return json.dumps([{"serie_id": s, "titulo": str(cat.loc[s, "serie_titulo"]),
                            "distribucion": str(cat.loc[s, "distribucion_titulo"]),
                            "frecuencia": str(cat.loc[s, "frecuencia"])} for s, _ in rows], ensure_ascii=False)

    def _api(self, args):
        params = json.loads(clave(args))
        for intento in range(3):
            try:
                r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
                if r.status_code in (200, 400):
                    return r.status_code, r.json()
            except Exception:
                pass
            time.sleep(1 + intento)
        return None, None

    def get_series(self, args):
        extra = set(args) - PARAMS_TOOL
        if extra or "ids" not in args:
            return json.dumps({"status": 400, "error": f"Parámetros no permitidos o faltantes: {sorted(extra) or ['ids']}"},
                              ensure_ascii=False)
        k = clave(args)
        fila = self.cache.execute("SELECT status, cuerpo FROM cache WHERE clave=?", (k,)).fetchone()
        if fila:
            status, js = fila[0], json.loads(fila[1])
        else:
            self.fallos_cache += 1
            status, js = self._api(args)
            if status is None:
                self.sin_red += 1
                return json.dumps({"status": None, "error": "no se pudo consultar la API"}, ensure_ascii=False)
            self.cache.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                               (k, status, json.dumps(js, ensure_ascii=False),
                                time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime())))
            self.cache.commit()
        if status != 200:
            return json.dumps({"status": status, "error": js.get("errors", js)}, ensure_ascii=False)
        sid = str(args["ids"])
        c = cat.loc[sid] if sid in cat.index else None
        return json.dumps({"status": 200, "serie_id": sid,
                           "titulo": None if c is None else str(c["serie_titulo"]),
                           "unidades": None if (c is None or pd.isna(c["serie_unidades"])) else c["serie_unidades"],
                           "frecuencia": None if c is None else str(c["frecuencia"]),
                           "data": js.get("data", [])}, ensure_ascii=False)

H = Herramientas()

# ---------- Evaluador (el mismo validado con el agente oráculo) ----------
def periodo(f, freq):
    f = "" if f is None else str(f)
    return f[:4] if freq == "anual" else f[:7]

def valor_fecha(tipo, freq, d, params):
    if tipo == "sin_datos" or not d or d[0][1] is None or d[-1][1] is None:
        return None, None
    if tipo == "ultimo_dato":
        return d[0][1], d[0][0]
    if tipo == "valor_puntual":
        return d[0][1], params["start_date"]
    return round(d[-1][1] * 100, 2), d[-1][0]

def cerca(a, b, tol_abs):
    return abs(a - b) <= max(tol_abs, 1e-3 * abs(b))

def puntuar(tipo, freq, g, sub, ids_ultima, objetivo):
    r = {"termino": sub is not None, "serie_ok": ids_ultima == objetivo}
    if sub is None:
        r.update(valor_ok=False, valor_flex=False, fecha_ok=False)
    elif tipo == "sin_datos":
        nulo = sub.get("valor") is None
        r.update(valor_ok=nulo, valor_flex=nulo, fecha_ok=sub.get("fecha") is None)
    else:
        v = sub.get("valor")
        num = isinstance(v, (int, float)) and not isinstance(v, bool)
        tol = 0.011 if tipo == "variacion_interanual" else 5e-5
        ok_e = bool(num and cerca(v, g["valor"], tol))
        ok_f = ok_e or bool(tipo == "variacion_interanual" and num and cerca(v, g["valor"] / 100, 1e-4))
        r.update(valor_ok=ok_e, valor_flex=ok_f,
                 fecha_ok=sub.get("fecha") is not None and periodo(sub["fecha"], freq) == periodo(g["fecha"], freq))
    r["correcta"] = bool(r["termino"] and r["serie_ok"] and r["valor_ok"] and r["fecha_ok"])
    r["correcta_flex"] = bool(r["termino"] and r["serie_ok"] and r["valor_flex"] and r["fecha_ok"])
    return r

def causa(r):
    if r["correcta"]:
        return "correcta"
    if not r["termino"]:
        return "no_termino"
    if not r["serie_ok"]:
        return "serie_equivocada"
    if not r["valor_ok"]:
        return "convencion_porcentaje" if r["valor_flex"] else "valor_equivocado"
    return "fecha_equivocada"

# ---------- Agente ----------
def generar(mensajes):
    ids = tokenizer.apply_chat_template(mensajes, tools=tools, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    n_in = ids.shape[1]
    if n_in > model.max_seq_length - MAX_NUEVOS:
        return None, n_in, 0, 0.0
    t = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NUEVOS, do_sample=False)
    nuevos = out[0][n_in:]
    return tokenizer.decode(nuevos, skip_special_tokens=False), n_in, len(nuevos), time.time() - t

RE_LLAMADA = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def episodio(pregunta, sistema):
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    log = {"pasos": 0, "n_search": 0, "n_get": 0, "formato_invalido": False, "sin_llamada": False,
           "contexto_excedido": False, "submit": None, "ids_ultima": None, "ultimo_texto": ""}
    for paso in range(1, MAX_PASOS + 1):
        log["pasos"] = paso
        with lf.start_as_current_observation(as_type="generation", name=f"generacion-{paso}",
                                             model=ETIQUETA_MODELO, input=msgs[-1]) as gen:
            txt, n_in, n_out, seg = generar(msgs)
            gen.update(output=txt, metadata={"tokens_entrada": n_in, "tokens_salida": n_out, "segundos": round(seg, 2)})
        if txt is None:
            log["contexto_excedido"] = True
            break
        txt = txt.replace("<|im_end|>", "").strip()
        log["ultimo_texto"] = txt
        m = RE_LLAMADA.search(txt)
        if not m:
            log["sin_llamada"] = True
            break
        try:
            llamada = json.loads(m.group(1))
            nombre, args = llamada["name"], llamada["arguments"]
            assert isinstance(args, dict)
        except Exception:
            log["formato_invalido"] = True
            break
        msgs.append({"role": "assistant", "content": "",
                     "tool_calls": [{"id": f"call_{paso}", "type": "function",
                                     "function": {"name": nombre, "arguments": args}}]})
        if nombre == "submit_answer":
            log["submit"] = args
            break
        with lf.start_as_current_observation(as_type="span", name=f"herramienta:{nombre}", input=args) as ts:
            if nombre == "search_series":
                log["n_search"] += 1
                q = args.get("query")
                res = H.search_series(q) if isinstance(q, str) else json.dumps({"status": 400, "error": "query debe ser texto"})
            elif nombre == "get_series":
                log["n_get"] += 1
                log["ids_ultima"] = str(args.get("ids"))
                res = H.get_series(args)
            else:
                res = json.dumps({"status": 400, "error": f"herramienta desconocida: {nombre}"}, ensure_ascii=False)
            ts.update(output=res[:1500])
        msgs.append({"role": "tool", "tool_call_id": f"call_{paso}", "name": nombre, "content": res})
    return log, msgs

# ---------- Preguntas ----------
Fv = pd.read_csv(f"{RUTA}/eval_{SPLIT}_final2.csv")
Fv = Fv[Fv["marca_var"].fillna("") != "var_sobre_tasa"]
if ESTILO == "coloquial":
    Fv = Fv[~Fv["coloquial_ambigua"]]
S = pd.read_csv(f"{RUTA}/specs_{SPLIT}_v2.csv")
S["idx"] = S.index
Fv["idx"] = Fv["spec_id"].str.split("-").str[1].astype(int)
Fv = Fv.merge(S[["idx", "data"]], on="idx", how="left")
if N_PRUEBA:
    Fv = Fv.sample(frac=1, random_state=4).groupby("tipo").head(int(np.ceil(N_PRUEBA / 4)))
Fv = Fv.reset_index(drop=True)
print(f"Preguntas: {len(Fv)} ({SPLIT}, estilo {ESTILO}) | por tipo: {Fv['tipo'].value_counts().to_dict()}")

# ---------- Retoma: se leen los resultados ya guardados ----------
if os.path.exists(ARCH_DRIVE) and not os.path.exists(ARCH_LOCAL):
    shutil.copy(ARCH_DRIVE, ARCH_LOCAL)
validas, hechos = [], set()
if os.path.exists(ARCH_LOCAL):
    for linea in open(ARCH_LOCAL):
        try:
            d = json.loads(linea)
            hechos.add(d["resultado"]["spec_id"])
            validas.append(linea if linea.endswith("\n") else linea + "\n")
        except Exception:
            pass                      # línea cortada por un corte de la sesión: se descarta
    with open(ARCH_LOCAL, "w") as f:
        f.writelines(validas)
print(f"Retomando: ya hechas {len(hechos)} de {len(Fv)}" if hechos else "Corrida nueva")

def guardar():
    shutil.copy(ARCH_LOCAL, ARCH_DRIVE)
    H.cache.commit()
    shutil.copy("/content/cache_api.db", f"{RUTA}/cache_api.db")

SCORE_ERR, nuevos, t_total = None, 0, time.time()
with open(ARCH_LOCAL, "a") as fout:
    for r in Fv.to_dict("records"):
        if r["spec_id"] in hechos:
            continue
        params, d_gold = json.loads(r["params"]), json.loads(r["data"])
        gv, gf = valor_fecha(r["tipo"], r["frecuencia"], d_gold, params)
        g = {"valor": gv, "fecha": gf}
        pregunta = r[ESTILO]
        t0 = time.time()
        with lf.start_as_current_observation(
                as_type="span", name="pregunta", input={"pregunta": pregunta},
                metadata={"modelo": ETIQUETA_MODELO, "prompt": PROMPT, "tipo": r["tipo"],
                          "estilo": ESTILO, "split": SPLIT, "spec_id": r["spec_id"]}) as span:
            tid = lf.get_current_trace_id()
            log, msgs = episodio(pregunta, SISTEMA)
            sc = puntuar(r["tipo"], r["frecuencia"], g, log["submit"], log["ids_ultima"], params["ids"])
            span.update(output={"submit": log["submit"], "correcta": sc["correcta"], "causa": causa(sc)})
            try:
                span.score_trace(name="correcta", value=float(sc["correcta"]))
            except Exception as e:
                SCORE_ERR = f"{type(e).__name__}: {str(e)[:100]}"
        fila = {"prompt": PROMPT, "spec_id": r["spec_id"], "tipo": r["tipo"], "frecuencia": r["frecuencia"],
                **sc, "causa": causa(sc),
                **{k: log[k] for k in ["pasos", "n_search", "n_get", "formato_invalido", "sin_llamada", "contexto_excedido"]},
                "segundos": round(time.time() - t0, 1), "trace_id": tid}
        traza = {"prompt": PROMPT, "spec_id": r["spec_id"], "tipo": r["tipo"], "pregunta": pregunta,
                 "messages": msgs, "log": log, "score": sc}
        fout.write(json.dumps({"resultado": fila, "traza": traza}, ensure_ascii=False, default=str) + "\n")
        fout.flush()
        nuevos += 1
        if nuevos % CADA == 0:
            guardar()
            print(f"  [{len(hechos) + nuevos}/{len(Fv)}] guardado en Drive | {(time.time() - t_total) / 60:.1f} min en esta sesión", flush=True)
guardar()
lf.flush()

# ---------- Resumen ----------
filas = [json.loads(l) for l in open(ARCH_LOCAL)]
R = pd.DataFrame([f["resultado"] for f in filas])
trazas = [f["traza"] for f in filas]

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h

print("\n" + "=" * 70)
print(f"### {CORRIDA} | prompt {PROMPT} | {SPLIT} | n = {len(R)} (de {len(Fv)} preguntas)")
k = int(R["correcta"].sum()); lo, hi = wilson(k, len(R))
print(f"correcta: {k}/{len(R)} = {k / len(R):.1%} (IC 95%: {lo:.0%} a {hi:.0%}) | acepta fracción: {R['correcta_flex'].mean():.0%} | "
      f"terminó con submit_answer: {R['termino'].mean():.0%} | serie correcta: {R['serie_ok'].mean():.0%}")
print("por tipo:", (R.groupby("tipo")["correcta"].mean() * 100).round(0).to_dict())
print("causas:", R["causa"].value_counts().to_dict())
print(f"pasos (prom.): {R['pasos'].mean():.1f} | JSON inválido: {int(R['formato_invalido'].sum())} | "
      f"sin llamada: {int(R['sin_llamada'].sum())} | contexto excedido: {int(R['contexto_excedido'].sum())} | "
      f"segundos por pregunta: {R['segundos'].mean():.1f} | tiempo de generación total: {R['segundos'].sum() / 60:.0f} min")
print("Scores en Langfuse:", "ok" if SCORE_ERR is None else f"NO funcionó ({SCORE_ERR})")
for t in R["trace_id"].head(2):
    try:
        print("URL de traza:", lf.get_trace_url(trace_id=t))
    except Exception as e:
        print("No pude armar la URL:", type(e).__name__)

R.to_csv(f"{RUTA}/{NOMBRE}.csv", index=False)
with open(f"{RUTA}/{NOMBRE}_episodios.jsonl", "w") as f:
    for t in trazas:
        f.write(json.dumps(t, ensure_ascii=False, default=str) + "\n")
print("Guardado:", f"{NOMBRE}.csv y _episodios.jsonl")